In [1]:
# ============================================================
# CELL 1 — FINAL MODEL TRAINING
# Imports + Global Configuration
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121

from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ------------------------------------------------------------
# 2. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"D:\MSC Bioinformatics\INTERNSHIP\BrainTumor")

DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_ROOT = DATA_ROOT / "processed" / "classification_224_rgb"
SPLITS_FILE = DATA_ROOT / "splits" / "classification_model_splits.csv"

MODELS_ROOT = PROJECT_ROOT / "models"
OUTPUTS_ROOT = PROJECT_ROOT / "outputs"

FINAL_MODEL_DIR = MODELS_ROOT / "final_densenet121"
FINAL_REPORT_DIR = OUTPUTS_ROOT / "reports" / "final_densenet121"
FINAL_FIGURE_DIR = OUTPUTS_ROOT / "figures" / "final_densenet121"

# Create required directories
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. DATA CONFIGURATION
# ------------------------------------------------------------

IMAGE_SIZE = (224, 224)
NUM_CHANNELS = 3
NUM_CLASSES = 4

CLASS_NAMES = [
    "glioma",
    "meningioma",
    "no_tumor",
    "pituitary"
]

CLASS_TO_INDEX = {
    "glioma": 0,
    "meningioma": 1,
    "no_tumor": 2,
    "pituitary": 3
}

INDEX_TO_CLASS = {
    0: "glioma",
    1: "meningioma",
    2: "no_tumor",
    3: "pituitary"
}

# ------------------------------------------------------------
# 4. TRAINING CONFIGURATION
# ------------------------------------------------------------

BATCH_SIZE = 32

HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 15

HEAD_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5

FINE_TUNE_LAST_N_LAYERS = 40

DROPOUT_RATE = 0.30

# ------------------------------------------------------------
# 5. GPU INFORMATION
# ------------------------------------------------------------

print("=" * 70)
print("FINAL DENSENET121 TRAINING — INITIALIZATION")
print("=" * 70)

print(f"\nTensorFlow version : {tf.__version__}")
print(f"NumPy version     : {np.__version__}")

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print(f"GPU detected       : YES")
    print(f"GPU devices        : {gpus}")
else:
    print("GPU detected       : NO — training will use CPU")

print("\nProject root:")
print(PROJECT_ROOT)

print("\nProcessed dataset:")
print(PROCESSED_ROOT)

print("\nSplit file:")
print(SPLITS_FILE)

print("\nFinal model directory:")
print(FINAL_MODEL_DIR)

print("\nFinal report directory:")
print(FINAL_REPORT_DIR)

print("\nFinal figure directory:")
print(FINAL_FIGURE_DIR)

# ------------------------------------------------------------
# 6. CONFIGURATION SUMMARY
# ------------------------------------------------------------

CONFIG = {
    "project_root": str(PROJECT_ROOT),
    "processed_root": str(PROCESSED_ROOT),
    "splits_file": str(SPLITS_FILE),
    "image_size": IMAGE_SIZE,
    "num_channels": NUM_CHANNELS,
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "class_to_index": CLASS_TO_INDEX,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "head_epochs": HEAD_EPOCHS,
    "fine_tune_epochs": FINE_TUNE_EPOCHS,
    "head_learning_rate": HEAD_LEARNING_RATE,
    "fine_tune_learning_rate": FINE_TUNE_LEARNING_RATE,
    "fine_tune_last_n_layers": FINE_TUNE_LAST_N_LAYERS,
    "dropout_rate": DROPOUT_RATE,
    "official_test_used": False,
    "training_protocol": (
        "Final DenseNet121 trained on all 5000 development images "
        "(4000 original train + 1000 validation); "
        "official 1000-image test set remains locked."
    ),
    "created_at": datetime.now().isoformat()
}

CONFIG_FILE = FINAL_REPORT_DIR / "FINAL_DENSENET121_TRAINING_CONFIGURATION.json"

with open(CONFIG_FILE, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=4)

print("\n" + "=" * 70)
print("CELL 1 COMPLETE")
print("=" * 70)
print(f"Configuration saved to:\n{CONFIG_FILE}")

FINAL DENSENET121 TRAINING — INITIALIZATION

TensorFlow version : 2.10.0
NumPy version     : 1.23.5
GPU detected       : YES
GPU devices        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Project root:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor

Processed dataset:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\data\processed\classification_224_rgb

Split file:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\data\splits\classification_model_splits.csv

Final model directory:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\models\final_densenet121

Final report directory:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\outputs\reports\final_densenet121

Final figure directory:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\outputs\figures\final_densenet121

CELL 1 COMPLETE
Configuration saved to:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\outputs\reports\final_densenet121\FINAL_DENSENET121_TRAINING_CONFIGURATION.json


In [2]:
# ============================================================
# CELL 2 — LOAD + VERIFY FIXED MODEL SPLIT
# ============================================================

print("=" * 70)
print("CELL 2 — FIXED SPLIT VERIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load the existing split file
# ------------------------------------------------------------

if not SPLITS_FILE.exists():
    raise FileNotFoundError(
        f"Split file not found:\n{SPLITS_FILE}"
    )

split_df = pd.read_csv(SPLITS_FILE)

print(f"\nSplit file loaded successfully.")
print(f"Shape: {split_df.shape}")

# ------------------------------------------------------------
# 2. Required columns
# ------------------------------------------------------------

required_columns = [
    "filename",
    "processed_filename",
    "tumor_label",
    "tumor_code",
    "plane_label",
    "plane_code",
    "split",
    "processed_path",
    "model_split",
    "class_index"
]

missing_columns = [
    col for col in required_columns
    if col not in split_df.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns: PASS")

# ------------------------------------------------------------
# 3. Verify model_split values
# ------------------------------------------------------------

model_split_counts = split_df["model_split"].value_counts().sort_index()

print("\nModel split counts:")
print(model_split_counts)

expected_split_counts = {
    "train": 4000,
    "validation": 1000,
    "official_test": 1000
}

for split_name, expected_count in expected_split_counts.items():

    actual_count = int(
        model_split_counts.get(split_name, 0)
    )

    if actual_count != expected_count:
        raise RuntimeError(
            f"Split mismatch for '{split_name}': "
            f"expected {expected_count}, found {actual_count}"
        )

print("\nSplit sizes: PASS")

# ------------------------------------------------------------
# 4. Verify total number of records
# ------------------------------------------------------------

if len(split_df) != 6000:
    raise RuntimeError(
        f"Expected 6000 records, found {len(split_df)}"
    )

print("Total classification records: 6000 — PASS")

# ------------------------------------------------------------
# 5. Verify development set
# ------------------------------------------------------------

development_df = split_df[
    split_df["model_split"].isin(["train", "validation"])
].copy()

official_test_df = split_df[
    split_df["model_split"] == "official_test"
].copy()

print("\nDevelopment set:")
print(f"  Total:      {len(development_df)}")
print(f"  Train:      {(development_df['model_split'] == 'train').sum()}")
print(f"  Validation: {(development_df['model_split'] == 'validation').sum()}")

print("\nOfficial test set:")
print(f"  Total:      {len(official_test_df)}")

if len(development_df) != 5000:
    raise RuntimeError(
        f"Development set must contain 5000 images, "
        f"found {len(development_df)}"
    )

if len(official_test_df) != 1000:
    raise RuntimeError(
        f"Official test set must contain 1000 images, "
        f"found {len(official_test_df)}"
    )

print("\nDevelopment/test partition: PASS")

# ------------------------------------------------------------
# 6. Class distribution — development set
# ------------------------------------------------------------

print("\nDevelopment-set class distribution:")

development_class_counts = (
    pd.crosstab(
        development_df["model_split"],
        development_df["tumor_label"]
    )
    .reindex(
        index=["train", "validation"],
        columns=CLASS_NAMES,
        fill_value=0
    )
)

print(development_class_counts)

# ------------------------------------------------------------
# 7. Class distribution — official test
# ------------------------------------------------------------

print("\nOfficial-test class distribution:")

test_class_counts = (
    official_test_df["tumor_label"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)

print(test_class_counts)

# Expected official test counts
expected_test_classes = {
    "glioma": 254,
    "meningioma": 306,
    "no_tumor": 140,
    "pituitary": 300
}

for class_name, expected_count in expected_test_classes.items():

    actual_count = int(
        test_class_counts.get(class_name, 0)
    )

    if actual_count != expected_count:
        raise RuntimeError(
            f"Official-test class mismatch for '{class_name}': "
            f"expected {expected_count}, found {actual_count}"
        )

print("\nOfficial-test class distribution: PASS")

# ------------------------------------------------------------
# 8. Verify class indices
# ------------------------------------------------------------

class_index_check = (
    split_df[["tumor_label", "class_index"]]
    .drop_duplicates()
    .sort_values(["class_index", "tumor_label"])
)

print("\nClass-index mapping:")
print(class_index_check.to_string(index=False))

for class_name, expected_index in CLASS_TO_INDEX.items():

    observed = split_df.loc[
        split_df["tumor_label"] == class_name,
        "class_index"
    ].unique()

    if len(observed) != 1 or int(observed[0]) != expected_index:
        raise RuntimeError(
            f"Class-index mismatch for {class_name}: "
            f"expected {expected_index}, found {observed}"
        )

print("\nClass-index mapping: PASS")

# ------------------------------------------------------------
# 9. Verify processed files exist
# ------------------------------------------------------------

print("\nChecking processed image files...")

missing_files = []

for path_value in development_df["processed_path"]:

    path = Path(path_value)

    if not path.exists():
        missing_files.append(str(path))

if missing_files:

    print(f"Missing processed files: {len(missing_files)}")

    for missing in missing_files[:10]:
        print(missing)

    raise RuntimeError(
        "One or more development images are missing."
    )

print(
    f"Development processed images found: "
    f"{len(development_df)} — PASS"
)

# ------------------------------------------------------------
# 10. Verify no overlap between development and test
# ------------------------------------------------------------

development_filenames = set(
    development_df["filename"].astype(str)
)

test_filenames = set(
    official_test_df["filename"].astype(str)
)

filename_overlap = (
    development_filenames &
    test_filenames
)

if filename_overlap:
    raise RuntimeError(
        f"Data leakage detected: "
        f"{len(filename_overlap)} overlapping filenames."
    )

print("Development/test filename overlap: 0 — PASS")

# ------------------------------------------------------------
# 11. Save verification record
# ------------------------------------------------------------

split_verification = pd.DataFrame([
    {
        "check": "total_records",
        "expected": 6000,
        "observed": len(split_df),
        "status": "PASS"
    },
    {
        "check": "development_records",
        "expected": 5000,
        "observed": len(development_df),
        "status": "PASS"
    },
    {
        "check": "official_test_records",
        "expected": 1000,
        "observed": len(official_test_df),
        "status": "PASS"
    },
    {
        "check": "train_records",
        "expected": 4000,
        "observed": int(
            (split_df["model_split"] == "train").sum()
        ),
        "status": "PASS"
    },
    {
        "check": "validation_records",
        "expected": 1000,
        "observed": int(
            (split_df["model_split"] == "validation").sum()
        ),
        "status": "PASS"
    },
    {
        "check": "official_test_used",
        "expected": 0,
        "observed": 0,
        "status": "PASS"
    },
    {
        "check": "development_test_filename_overlap",
        "expected": 0,
        "observed": len(filename_overlap),
        "status": "PASS"
    }
])

verification_file = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_SPLIT_VERIFICATION.csv"
)

split_verification.to_csv(
    verification_file,
    index=False
)

# ------------------------------------------------------------
# 12. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 2 COMPLETE — ALL CHECKS PASSED")
print("=" * 70)

print("\nFINAL TRAINING DATA:")
print("  Development images : 5,000")
print("  Training subset     : 4,000")
print("  Validation subset   : 1,000")

print("\nLOCKED TEST DATA:")
print("  Official test       : 1,000")
print("  Used in this cell   : NO")
print("  Used for training   : NO")
print("  Used for selection  : NO")

print(f"\nVerification saved to:")
print(verification_file)

CELL 2 — FIXED SPLIT VERIFICATION

Split file loaded successfully.
Shape: (6000, 21)

Required columns: PASS

Model split counts:
model_split
official_test    1000
train            4000
validation       1000
Name: count, dtype: int64

Split sizes: PASS
Total classification records: 6000 — PASS

Development set:
  Total:      5000
  Train:      4000
  Validation: 1000

Official test set:
  Total:      1000

Development/test partition: PASS

Development-set class distribution:
tumor_label  glioma  meningioma  no_tumor  pituitary
model_split                                         
train           918        1063       854       1165
validation      229         266       213        292

Official-test class distribution:
tumor_label
glioma        254
meningioma    306
no_tumor      140
pituitary     300
Name: count, dtype: int64

Official-test class distribution: PASS

Class-index mapping:
tumor_label  class_index
     glioma            0
 meningioma            1
   no_tumor            2
 

In [3]:
# ============================================================
# CELL 3 — FINAL DEVELOPMENT DATASET PIPELINE
# ============================================================

print("=" * 70)
print("CELL 3 — BUILD FINAL 5,000-IMAGE DEVELOPMENT PIPELINE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Create final development dataframe
# ------------------------------------------------------------

final_development_df = split_df[
    split_df["model_split"].isin(["train", "validation"])
].copy()

final_development_df = final_development_df.reset_index(drop=True)

# ------------------------------------------------------------
# 2. Verify exactly 5,000 images
# ------------------------------------------------------------

if len(final_development_df) != 5000:
    raise RuntimeError(
        f"Expected 5,000 development images, "
        f"found {len(final_development_df)}"
    )

print(f"\nDevelopment images: {len(final_development_df)}")

# ------------------------------------------------------------
# 3. Verify class mapping
# ------------------------------------------------------------

final_development_df["label_index"] = (
    final_development_df["tumor_label"]
    .map(CLASS_TO_INDEX)
)

if final_development_df["label_index"].isna().any():
    raise RuntimeError(
        "One or more development images have an invalid class label."
    )

final_development_df["label_index"] = (
    final_development_df["label_index"].astype(int)
)

print("\nDevelopment class distribution:")

development_counts = (
    final_development_df["tumor_label"]
    .value_counts()
    .reindex(CLASS_NAMES)
)

print(development_counts)

# ------------------------------------------------------------
# 4. Verify processed image paths
# ------------------------------------------------------------

final_development_df["processed_path"] = (
    final_development_df["processed_path"]
    .astype(str)
)

missing_paths = [
    path
    for path in final_development_df["processed_path"]
    if not Path(path).exists()
]

if missing_paths:
    raise RuntimeError(
        f"{len(missing_paths)} processed images are missing."
    )

print(
    f"\nProcessed image files verified: "
    f"{len(final_development_df)} — PASS"
)

# ------------------------------------------------------------
# 5. Create TensorFlow loading function
# ------------------------------------------------------------

def load_final_image(path, label):
    """
    Load a preprocessed 224x224 RGB PNG image.

    Images were already:
    - converted to RGB
    - resized with aspect-ratio preservation
    - padded symmetrically
    - normalized to [0, 1]
    """

    image = tf.io.read_file(path)

    image = tf.image.decode_png(
        image,
        channels=3
    )

    image = tf.image.convert_image_dtype(
        image,
        dtype=tf.float32
    )

    image = tf.ensure_shape(
        image,
        [IMAGE_SIZE[0], IMAGE_SIZE[1], NUM_CHANNELS]
    )

    label = tf.cast(
        label,
        tf.int32
    )

    return image, label


# ------------------------------------------------------------
# 6. Convert dataframe to TensorFlow tensors
# ------------------------------------------------------------

image_paths = final_development_df[
    "processed_path"
].values

labels = final_development_df[
    "label_index"
].values.astype(np.int32)

# ------------------------------------------------------------
# 7. Build TensorFlow dataset
# ------------------------------------------------------------

final_train_dataset = tf.data.Dataset.from_tensor_slices(
    (image_paths, labels)
)

final_train_dataset = final_train_dataset.map(
    load_final_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

final_train_dataset = final_train_dataset.shuffle(
    buffer_size=len(final_development_df),
    seed=SEED,
    reshuffle_each_iteration=True
)

final_train_dataset = final_train_dataset.batch(
    BATCH_SIZE,
    drop_remainder=False
)

final_train_dataset = final_train_dataset.prefetch(
    tf.data.AUTOTUNE
)

# ------------------------------------------------------------
# 8. Inspect one batch
# ------------------------------------------------------------

sample_images, sample_labels = next(
    iter(final_train_dataset)
)

print("\nSample batch:")
print(f"  Image tensor shape : {sample_images.shape}")
print(f"  Image dtype        : {sample_images.dtype}")
print(f"  Label tensor shape : {sample_labels.shape}")
print(f"  Label dtype        : {sample_labels.dtype}")

print(
    f"  Pixel minimum      : "
    f"{float(tf.reduce_min(sample_images)):.6f}"
)

print(
    f"  Pixel maximum      : "
    f"{float(tf.reduce_max(sample_images)):.6f}"
)

# ------------------------------------------------------------
# 9. Validate image properties
# ------------------------------------------------------------

if sample_images.shape[1:] != (
    IMAGE_SIZE[0],
    IMAGE_SIZE[1],
    NUM_CHANNELS
):
    raise RuntimeError(
        f"Unexpected image shape: {sample_images.shape}"
    )

if sample_images.dtype != tf.float32:
    raise RuntimeError(
        f"Unexpected image dtype: {sample_images.dtype}"
    )

pixel_min = float(tf.reduce_min(sample_images))
pixel_max = float(tf.reduce_max(sample_images))

if pixel_min < 0.0 or pixel_max > 1.0:
    raise RuntimeError(
        "Pixel values are outside the expected [0, 1] range."
    )

# ------------------------------------------------------------
# 10. Save development dataset record
# ------------------------------------------------------------

development_record = final_development_df.copy()

development_record["dataset_role"] = "final_development"
development_record["official_test_used"] = False

development_record_file = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_DEVELOPMENT_DATASET_RECORD.csv"
)

development_record.to_csv(
    development_record_file,
    index=False
)

# ------------------------------------------------------------
# 11. Save pipeline configuration
# ------------------------------------------------------------

pipeline_config = {
    "dataset": "BRISC2025 classification",
    "development_samples": int(len(final_development_df)),
    "original_train_samples": int(
        (final_development_df["model_split"] == "train").sum()
    ),
    "original_validation_samples": int(
        (final_development_df["model_split"] == "validation").sum()
    ),
    "official_test_samples": int(len(official_test_df)),
    "official_test_used": False,
    "image_size": list(IMAGE_SIZE),
    "channels": NUM_CHANNELS,
    "normalization": "[0, 1]",
    "batch_size": BATCH_SIZE,
    "shuffle": True,
    "seed": SEED,
    "augmentation": False,
    "class_names": CLASS_NAMES,
    "class_to_index": CLASS_TO_INDEX
}

pipeline_config_file = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_PIPELINE_CONFIGURATION.json"
)

with open(
    pipeline_config_file,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        pipeline_config,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 12. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 3 COMPLETE — DEVELOPMENT PIPELINE READY")
print("=" * 70)

print("\nFINAL DEVELOPMENT DATA:")
print("  Total images       : 5,000")
print("  Original train     : 4,000")
print("  Original validation: 1,000")

print("\nOFFICIAL TEST:")
print("  Images             : 1,000")
print("  Used               : NO")
print("  Status             : LOCKED")

print("\nPIPELINE:")
print("  Image size         : 224 × 224 × 3")
print("  Data range         : [0, 1]")
print(f"  Batch size         : {BATCH_SIZE}")
print("  Augmentation       : None")
print(f"  Random seed        : {SEED}")

print(f"\nDataset record saved to:")
print(development_record_file)

print(f"\nPipeline configuration saved to:")
print(pipeline_config_file)

CELL 3 — BUILD FINAL 5,000-IMAGE DEVELOPMENT PIPELINE

Development images: 5000

Development class distribution:
tumor_label
glioma        1147
meningioma    1329
no_tumor      1067
pituitary     1457
Name: count, dtype: int64

Processed image files verified: 5000 — PASS

Sample batch:
  Image tensor shape : (32, 224, 224, 3)
  Image dtype        : <dtype: 'float32'>
  Label tensor shape : (32,)
  Label dtype        : <dtype: 'int32'>
  Pixel minimum      : 0.000000
  Pixel maximum      : 1.000000

CELL 3 COMPLETE — DEVELOPMENT PIPELINE READY

FINAL DEVELOPMENT DATA:
  Total images       : 5,000
  Original train     : 4,000
  Original validation: 1,000

OFFICIAL TEST:
  Images             : 1,000
  Used               : NO
  Status             : LOCKED

PIPELINE:
  Image size         : 224 × 224 × 3
  Data range         : [0, 1]
  Batch size         : 32
  Augmentation       : None
  Random seed        : 42

Dataset record saved to:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\outputs\re

In [4]:
# ============================================================
# CELL 4 — FINAL CLASS WEIGHTS
# ============================================================

print("=" * 70)
print("CELL 4 — CALCULATE CLASS WEIGHTS FROM 5,000 DEVELOPMENT IMAGES")
print("=" * 70)

# ------------------------------------------------------------
# 1. Extract development labels
# ------------------------------------------------------------

development_labels = final_development_df["label_index"].values

# ------------------------------------------------------------
# 2. Calculate balanced class weights
# ------------------------------------------------------------

class_indices = np.arange(NUM_CLASSES)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=class_indices,
    y=development_labels
)

class_weights = {
    int(class_index): float(weight)
    for class_index, weight
    in zip(class_indices, class_weights_array)
}

# ------------------------------------------------------------
# 3. Display weights
# ------------------------------------------------------------

print("\nFinal development class weights:")

for index in class_indices:
    print(
        f"  {index} — {INDEX_TO_CLASS[index]:12s}: "
        f"{class_weights[index]:.6f}"
    )

# ------------------------------------------------------------
# 4. Verify class-weight calculation
# ------------------------------------------------------------

expected_counts = (
    final_development_df["label_index"]
    .value_counts()
    .sort_index()
)

expected_formula_weights = (
    len(development_labels) /
    (
        NUM_CLASSES *
        expected_counts
    )
)

for index in class_indices:

    calculated = class_weights[index]
    expected = float(expected_formula_weights[index])

    if not np.isclose(calculated, expected, rtol=1e-6):
        raise RuntimeError(
            f"Class-weight calculation mismatch for "
            f"class {index}."
        )

print("\nClass-weight formula verification: PASS")

# ------------------------------------------------------------
# 5. Create readable class-weight table
# ------------------------------------------------------------

class_weight_table = pd.DataFrame({
    "class_index": class_indices,
    "class_name": [
        INDEX_TO_CLASS[i]
        for i in class_indices
    ],
    "sample_count": [
        int(expected_counts[i])
        for i in class_indices
    ],
    "class_weight": [
        class_weights[i]
        for i in class_indices
    ]
})

print("\nClass-weight table:")
print(
    class_weight_table.to_string(index=False)
)

# ------------------------------------------------------------
# 6. Save class weights
# ------------------------------------------------------------

class_weights_file = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_CLASS_WEIGHTS.csv"
)

class_weight_table.to_csv(
    class_weights_file,
    index=False
)

# ------------------------------------------------------------
# 7. Save JSON version
# ------------------------------------------------------------

class_weights_json = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_CLASS_WEIGHTS.json"
)

with open(
    class_weights_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            INDEX_TO_CLASS[i]: class_weights[i]
            for i in class_indices
        },
        f,
        indent=4
    )

# ------------------------------------------------------------
# 8. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 4 COMPLETE — CLASS WEIGHTS READY")
print("=" * 70)

print(f"\nCSV saved to:")
print(class_weights_file)

print(f"\nJSON saved to:")
print(class_weights_json)

CELL 4 — CALCULATE CLASS WEIGHTS FROM 5,000 DEVELOPMENT IMAGES

Final development class weights:
  0 — glioma      : 1.089799
  1 — meningioma  : 0.940557
  2 — no_tumor    : 1.171509
  3 — pituitary   : 0.857927

Class-weight formula verification: PASS

Class-weight table:
 class_index class_name  sample_count  class_weight
           0     glioma          1147      1.089799
           1 meningioma          1329      0.940557
           2   no_tumor          1067      1.171509
           3  pituitary          1457      0.857927

CELL 4 COMPLETE — CLASS WEIGHTS READY

CSV saved to:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\outputs\reports\final_densenet121\FINAL_DENSENET121_CLASS_WEIGHTS.csv

JSON saved to:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\outputs\reports\final_densenet121\FINAL_DENSENET121_CLASS_WEIGHTS.json


In [5]:
# ============================================================
# CELL 5 — BUILD FINAL DENSENET121 MODEL
# ============================================================

print("=" * 70)
print("CELL 5 — BUILD FINAL DENSENET121")
print("=" * 70)

# ------------------------------------------------------------
# 1. Clear any previous Keras model state
# ------------------------------------------------------------

tf.keras.backend.clear_session()

# Re-establish reproducibility after clearing the session
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ------------------------------------------------------------
# 2. Create DenseNet121 backbone
# ------------------------------------------------------------

print("\nLoading DenseNet121 ImageNet backbone...")

base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=(
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        NUM_CHANNELS
    )
)

print("ImageNet weights loaded successfully.")

# ------------------------------------------------------------
# 3. Freeze backbone for initial training
# ------------------------------------------------------------

base_model.trainable = False

# ------------------------------------------------------------
# 4. Build final classification model
# ------------------------------------------------------------

inputs = keras.Input(
    shape=(
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        NUM_CHANNELS
    ),
    name="mri_input"
)

x = base_model(
    inputs,
    training=False
)

x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = layers.BatchNormalization(
    name="classifier_batch_normalization"
)(x)

x = layers.Dropout(
    DROPOUT_RATE,
    name="classifier_dropout"
)(x)

x = layers.Dense(
    256,
    activation="relu",
    name="classifier_dense"
)(x)

x = layers.Dropout(
    DROPOUT_RATE,
    name="classifier_dropout_2"
)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="predictions"
)(x)

final_model = keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="Final_DenseNet121_BRISC2025"
)

# ------------------------------------------------------------
# 5. Compile model for head training
# ------------------------------------------------------------

final_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=HEAD_LEARNING_RATE
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

# ------------------------------------------------------------
# 6. Model summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL ARCHITECTURE")
print("=" * 70)

final_model.summary()

# ------------------------------------------------------------
# 7. Parameter counts
# ------------------------------------------------------------

total_params = final_model.count_params()

trainable_params = sum(
    np.prod(variable.shape)
    for variable in final_model.trainable_variables
)

non_trainable_params = sum(
    np.prod(variable.shape)
    for variable in final_model.non_trainable_variables
)

print("\n" + "=" * 70)
print("PARAMETER SUMMARY")
print("=" * 70)

print(f"Total parameters      : {total_params:,}")
print(f"Trainable parameters  : {trainable_params:,}")
print(f"Non-trainable params  : {non_trainable_params:,}")

# ------------------------------------------------------------
# 8. Verify backbone is frozen
# ------------------------------------------------------------

backbone_trainable = sum(
    1 for layer in base_model.layers
    if layer.trainable
)

backbone_total = len(base_model.layers)

print("\nBackbone status:")
print(f"  Total backbone layers    : {backbone_total}")
print(f"  Trainable backbone layers: {backbone_trainable}")
print(f"  Frozen backbone layers   : {backbone_total - backbone_trainable}")

if backbone_trainable != 0:
    raise RuntimeError(
        "DenseNet121 backbone was not completely frozen."
    )

print("\nBackbone freeze verification: PASS")

# ------------------------------------------------------------
# 9. Verify output shape
# ------------------------------------------------------------

if final_model.output_shape != (None, NUM_CLASSES):
    raise RuntimeError(
        f"Unexpected model output shape: "
        f"{final_model.output_shape}"
    )

print(
    f"Model output shape: "
    f"{final_model.output_shape} — PASS"
)

# ------------------------------------------------------------
# 10. Save architecture summary
# ------------------------------------------------------------

architecture_file = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_ARCHITECTURE.txt"
)

with open(
    architecture_file,
    "w",
    encoding="utf-8"
) as f:

    final_model.summary(
        print_fn=lambda line: f.write(line + "\n")
    )

# ------------------------------------------------------------
# 11. Save model architecture JSON
# ------------------------------------------------------------

architecture_json_file = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_ARCHITECTURE.json"
)

with open(
    architecture_json_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        final_model.to_json()
    )

# ------------------------------------------------------------
# 12. Save parameter record
# ------------------------------------------------------------

parameter_record = pd.DataFrame([{
    "model": "DenseNet121",
    "input_height": IMAGE_SIZE[0],
    "input_width": IMAGE_SIZE[1],
    "input_channels": NUM_CHANNELS,
    "num_classes": NUM_CLASSES,
    "total_parameters": int(total_params),
    "trainable_parameters_head_stage": int(trainable_params),
    "non_trainable_parameters_head_stage": int(non_trainable_params),
    "backbone_trainable": False,
    "imagenet_initialization": True,
    "official_test_used": False,
    "status": "PASS"
}])

parameter_file = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_PARAMETER_RECORD.csv"
)

parameter_record.to_csv(
    parameter_file,
    index=False
)

# ------------------------------------------------------------
# 13. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 5 COMPLETE — FINAL DENSENET121 READY")
print("=" * 70)

print("\nModel:")
print("  DenseNet121")
print("  ImageNet initialization: YES")
print("  Backbone frozen: YES")
print(f"  Output classes: {NUM_CLASSES}")

print("\nOfficial test:")
print("  Used: NO")
print("  Status: LOCKED")

print(f"\nArchitecture saved to:")
print(architecture_file)

print(f"\nArchitecture JSON saved to:")
print(architecture_json_file)

print(f"\nParameter record saved to:")
print(parameter_file)

CELL 5 — BUILD FINAL DENSENET121

Loading DenseNet121 ImageNet backbone...
ImageNet weights loaded successfully.

MODEL ARCHITECTURE
Model: "Final_DenseNet121_BRISC2025"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mri_input (InputLayer)      [(None, 224, 224, 3)]     0         
                                                                 
 densenet121 (Functional)    (None, 7, 7, 1024)        7037504   
                                                                 
 global_average_pooling (Glo  (None, 1024)             0         
 balAveragePooling2D)                                            
                                                                 
 classifier_batch_normalizat  (None, 1024)             4096      
 ion (BatchNormalization)                                        
                                                                 
 classifier_dropout (Dropout  (None, 1

In [6]:
# ============================================================
# CELL 6 — FINAL DENSENET121: HEAD TRAINING
# ============================================================

print("=" * 70)
print("CELL 6 — TRAIN FINAL DENSENET121 CLASSIFICATION HEAD")
print("=" * 70)

# ------------------------------------------------------------
# 1. Training callbacks
# ------------------------------------------------------------

HEAD_CHECKPOINT = (
    FINAL_MODEL_DIR /
    "densenet121_final_head_best.weights.h5"
)

HEAD_LOG = (
    FINAL_REPORT_DIR /
    "densenet121_final_head_training_log.csv"
)

head_callbacks = [

    keras.callbacks.ModelCheckpoint(
        filepath=str(HEAD_CHECKPOINT),
        monitor="loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        filename=str(HEAD_LOG),
        append=False
    )
]

# ------------------------------------------------------------
# 2. Confirm model is in head-training mode
# ------------------------------------------------------------

base_model.trainable = False

# Recompile after confirming trainability
final_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=HEAD_LEARNING_RATE
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

trainable_count = sum(
    np.prod(variable.shape)
    for variable in final_model.trainable_variables
)

print("\nTraining configuration:")
print(f"  Model                 : DenseNet121")
print(f"  Development images    : 5,000")
print(f"  Epochs                : {HEAD_EPOCHS}")
print(f"  Batch size            : {BATCH_SIZE}")
print(f"  Learning rate         : {HEAD_LEARNING_RATE}")
print(f"  Class weights         : YES")
print(f"  Data augmentation     : NO")
print(f"  Official test used    : NO")
print(f"  Trainable parameters  : {trainable_count:,}")

# ------------------------------------------------------------
# 3. Verify class weights
# ------------------------------------------------------------

if set(class_weights.keys()) != set(range(NUM_CLASSES)):
    raise RuntimeError(
        "Class weights do not contain all four class indices."
    )

# ------------------------------------------------------------
# 4. Train
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STARTING HEAD TRAINING")
print("=" * 70)

head_history = final_model.fit(
    final_train_dataset,
    epochs=HEAD_EPOCHS,
    class_weight=class_weights,
    callbacks=head_callbacks,
    verbose=1
)

# ------------------------------------------------------------
# 5. Convert history to dataframe
# ------------------------------------------------------------

head_history_df = pd.DataFrame(
    head_history.history
)

head_history_df.insert(
    0,
    "epoch",
    np.arange(
        1,
        len(head_history_df) + 1
    )
)

head_history_file = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_HEAD_TRAINING_HISTORY.csv"
)

head_history_df.to_csv(
    head_history_file,
    index=False
)

# ------------------------------------------------------------
# 6. Find best epoch
# ------------------------------------------------------------

best_epoch_index = (
    head_history_df["loss"].idxmin()
)

best_epoch = int(
    head_history_df.loc[
        best_epoch_index,
        "epoch"
    ]
)

best_loss = float(
    head_history_df.loc[
        best_epoch_index,
        "loss"
    ]
)

best_accuracy = float(
    head_history_df.loc[
        best_epoch_index,
        "accuracy"
    ]
)

# ------------------------------------------------------------
# 7. Save training-stage record
# ------------------------------------------------------------

head_record = pd.DataFrame([{
    "model": "DenseNet121",
    "training_stage": "classification_head",
    "development_samples": 5000,
    "epochs_requested": HEAD_EPOCHS,
    "epochs_completed": len(head_history_df),
    "batch_size": BATCH_SIZE,
    "learning_rate": HEAD_LEARNING_RATE,
    "class_weights_used": True,
    "augmentation_used": False,
    "best_epoch": best_epoch,
    "best_loss": best_loss,
    "best_accuracy": best_accuracy,
    "official_test_used": False,
    "status": "PASS"
}])

head_record_file = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_HEAD_TRAINING_RECORD.csv"
)

head_record.to_csv(
    head_record_file,
    index=False
)

# ------------------------------------------------------------
# 8. Verify checkpoint
# ------------------------------------------------------------

if not HEAD_CHECKPOINT.exists():
    raise RuntimeError(
        "Best head-training weights were not saved."
    )

checkpoint_size_mb = (
    HEAD_CHECKPOINT.stat().st_size /
    (1024 ** 2)
)

# ------------------------------------------------------------
# 9. Final output
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6 COMPLETE — HEAD TRAINING FINISHED")
print("=" * 70)

print(f"\nEpochs completed : {len(head_history_df)}")
print(f"Best epoch       : {best_epoch}")
print(f"Best loss        : {best_loss:.6f}")
print(f"Best accuracy    : {best_accuracy:.6f}")

print(f"\nBest weights saved:")
print(HEAD_CHECKPOINT)

print(f"Checkpoint size  : {checkpoint_size_mb:.2f} MB")

print(f"\nHistory saved:")
print(head_history_file)

print(f"\nTraining record saved:")
print(head_record_file)

print("\nOfficial test used: NO")

CELL 6 — TRAIN FINAL DENSENET121 CLASSIFICATION HEAD

Training configuration:
  Model                 : DenseNet121
  Development images    : 5,000
  Epochs                : 10
  Batch size            : 32
  Learning rate         : 0.001
  Class weights         : YES
  Data augmentation     : NO
  Official test used    : NO
  Trainable parameters  : 265,476

STARTING HEAD TRAINING
Epoch 1/10
157/157 [==============================] - ETA: 0s - loss: 0.4055 - accuracy: 0.8450
Epoch 1: loss improved from inf to 0.40553, saving model to D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\models\final_densenet121\densenet121_final_head_best.weights.h5
157/157 [==============================] - 37s 161ms/step - loss: 0.4055 - accuracy: 0.8450 - lr: 0.0010
Epoch 2/10
156/157 [============================>.] - ETA: 0s - loss: 0.2234 - accuracy: 0.9135
Epoch 2: loss improved from 0.40553 to 0.22436, saving model to D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\models\final_densenet121\densenet121_final_

In [7]:
# ============================================================
# CELL 7 — FINAL DENSENET121: FINE-TUNING
# ============================================================

print("=" * 70)
print("CELL 7 — FINE-TUNE FINAL DENSENET121")
print("=" * 70)

# ------------------------------------------------------------
# 1. Restore best head-training weights
# ------------------------------------------------------------

if not HEAD_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Head-training checkpoint not found:\n{HEAD_CHECKPOINT}"
    )

print("\nLoading best classification-head weights...")

final_model.load_weights(
    str(HEAD_CHECKPOINT)
)

print("Best head-training weights loaded successfully.")

# ------------------------------------------------------------
# 2. Freeze all layers first
# ------------------------------------------------------------

base_model.trainable = True

# Freeze every layer initially
for layer in base_model.layers:
    layer.trainable = False

# ------------------------------------------------------------
# 3. Unfreeze the final N layers
# ------------------------------------------------------------

layers_to_unfreeze = base_model.layers[
    -FINE_TUNE_LAST_N_LAYERS:
]

for layer in layers_to_unfreeze:

    # Keep BatchNormalization frozen for stable statistics
    if not isinstance(
        layer,
        layers.BatchNormalization
    ):
        layer.trainable = True

# ------------------------------------------------------------
# 4. Count trainable layers
# ------------------------------------------------------------

trainable_backbone_layers = [
    layer
    for layer in base_model.layers
    if layer.trainable
]

frozen_backbone_layers = [
    layer
    for layer in base_model.layers
    if not layer.trainable
]

print("\nFine-tuning configuration:")
print(f"  Total backbone layers      : {len(base_model.layers)}")
print(f"  Last layers considered    : {FINE_TUNE_LAST_N_LAYERS}")
print(
    f"  Trainable backbone layers : "
    f"{len(trainable_backbone_layers)}"
)
print(
    f"  Frozen backbone layers    : "
    f"{len(frozen_backbone_layers)}"
)

# ------------------------------------------------------------
# 5. Display layers being fine-tuned
# ------------------------------------------------------------

print("\nLayers being fine-tuned:")

for layer in trainable_backbone_layers:
    print(
        f"  {layer.name:45s} "
        f"{layer.__class__.__name__}"
    )

# ------------------------------------------------------------
# 6. Recompile with small learning rate
# ------------------------------------------------------------

final_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=FINE_TUNE_LEARNING_RATE
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

# ------------------------------------------------------------
# 7. Parameter counts
# ------------------------------------------------------------

total_params_ft = final_model.count_params()

trainable_params_ft = sum(
    np.prod(variable.shape)
    for variable in final_model.trainable_variables
)

non_trainable_params_ft = sum(
    np.prod(variable.shape)
    for variable in final_model.non_trainable_variables
)

print("\nParameter counts:")
print(f"  Total parameters     : {total_params_ft:,}")
print(f"  Trainable parameters : {trainable_params_ft:,}")
print(f"  Frozen parameters    : {non_trainable_params_ft:,}")

# ------------------------------------------------------------
# 8. Fine-tuning callbacks
# ------------------------------------------------------------

FINE_TUNE_CHECKPOINT = (
    FINAL_MODEL_DIR /
    "densenet121_final_finetuned_best.weights.h5"
)

FINE_TUNE_LOG = (
    FINAL_REPORT_DIR /
    "densenet121_final_finetuning_log.csv"
)

fine_tune_callbacks = [

    keras.callbacks.ModelCheckpoint(
        filepath=str(FINE_TUNE_CHECKPOINT),
        monitor="loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        filename=str(FINE_TUNE_LOG),
        append=False
    )
]

# ------------------------------------------------------------
# 9. Start fine-tuning
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STARTING DENSENET121 FINE-TUNING")
print("=" * 70)

print(f"\nEpochs         : {FINE_TUNE_EPOCHS}")
print(f"Learning rate  : {FINE_TUNE_LEARNING_RATE}")
print(f"Batch size     : {BATCH_SIZE}")
print("Class weights  : YES")
print("Official test  : NO")

fine_tune_history = final_model.fit(
    final_train_dataset,
    epochs=FINE_TUNE_EPOCHS,
    class_weight=class_weights,
    callbacks=fine_tune_callbacks,
    verbose=1
)

# ------------------------------------------------------------
# 10. Convert history to dataframe
# ------------------------------------------------------------

fine_tune_history_df = pd.DataFrame(
    fine_tune_history.history
)

fine_tune_history_df.insert(
    0,
    "epoch",
    np.arange(
        1,
        len(fine_tune_history_df) + 1
    )
)

fine_tune_history_file = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_FINETUNING_HISTORY.csv"
)

fine_tune_history_df.to_csv(
    fine_tune_history_file,
    index=False
)

# ------------------------------------------------------------
# 11. Find best fine-tuning epoch
# ------------------------------------------------------------

best_ft_epoch_index = (
    fine_tune_history_df["loss"].idxmin()
)

best_ft_epoch = int(
    fine_tune_history_df.loc[
        best_ft_epoch_index,
        "epoch"
    ]
)

best_ft_loss = float(
    fine_tune_history_df.loc[
        best_ft_epoch_index,
        "loss"
    ]
)

best_ft_accuracy = float(
    fine_tune_history_df.loc[
        best_ft_epoch_index,
        "accuracy"
    ]
)

# ------------------------------------------------------------
# 12. Save fine-tuning record
# ------------------------------------------------------------

fine_tune_record = pd.DataFrame([{
    "model": "DenseNet121",
    "training_stage": "fine_tuning",
    "development_samples": 5000,
    "epochs_requested": FINE_TUNE_EPOCHS,
    "epochs_completed": len(fine_tune_history_df),
    "batch_size": BATCH_SIZE,
    "learning_rate": FINE_TUNE_LEARNING_RATE,
    "layers_considered_for_unfreezing": FINE_TUNE_LAST_N_LAYERS,
    "trainable_backbone_layers": len(trainable_backbone_layers),
    "class_weights_used": True,
    "batch_normalization_layers_frozen": True,
    "augmentation_used": False,
    "best_epoch": best_ft_epoch,
    "best_loss": best_ft_loss,
    "best_accuracy": best_ft_accuracy,
    "official_test_used": False,
    "status": "PASS"
}])

fine_tune_record_file = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_FINETUNING_RECORD.csv"
)

fine_tune_record.to_csv(
    fine_tune_record_file,
    index=False
)

# ------------------------------------------------------------
# 13. Verify checkpoint
# ------------------------------------------------------------

if not FINE_TUNE_CHECKPOINT.exists():
    raise RuntimeError(
        "Fine-tuning checkpoint was not saved."
    )

checkpoint_size_mb = (
    FINE_TUNE_CHECKPOINT.stat().st_size /
    (1024 ** 2)
)

# ------------------------------------------------------------
# 14. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 7 COMPLETE — FINE-TUNING FINISHED")
print("=" * 70)

print(f"\nEpochs completed : {len(fine_tune_history_df)}")
print(f"Best epoch       : {best_ft_epoch}")
print(f"Best loss        : {best_ft_loss:.6f}")
print(f"Best accuracy    : {best_ft_accuracy:.6f}")

print(f"\nBest fine-tuned weights:")
print(FINE_TUNE_CHECKPOINT)

print(f"Checkpoint size  : {checkpoint_size_mb:.2f} MB")

print(f"\nHistory saved:")
print(fine_tune_history_file)

print(f"\nTraining record saved:")
print(fine_tune_record_file)

print("\nOfficial test used: NO")

CELL 7 — FINE-TUNE FINAL DENSENET121

Loading best classification-head weights...
Best head-training weights loaded successfully.

Fine-tuning configuration:
  Total backbone layers      : 427
  Last layers considered    : 40
  Trainable backbone layers : 29
  Frozen backbone layers    : 398

Layers being fine-tuned:
  conv5_block11_1_relu                          Activation
  conv5_block11_2_conv                          Conv2D
  conv5_block11_concat                          Concatenate
  conv5_block12_0_relu                          Activation
  conv5_block12_1_conv                          Conv2D
  conv5_block12_1_relu                          Activation
  conv5_block12_2_conv                          Conv2D
  conv5_block12_concat                          Concatenate
  conv5_block13_0_relu                          Activation
  conv5_block13_1_conv                          Conv2D
  conv5_block13_1_relu                          Activation
  conv5_block13_2_conv                        

In [8]:
# ============================================================
# CELL 8 — SAVE + VERIFY FINAL DENSENET121
# ============================================================

print("=" * 70)
print("CELL 8 — SAVE + VERIFY FINAL DENSENET121")
print("=" * 70)

# ------------------------------------------------------------
# 1. Restore best fine-tuning weights
# ------------------------------------------------------------

if not FINE_TUNE_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Fine-tuning checkpoint not found:\n{FINE_TUNE_CHECKPOINT}"
    )

print("\nLoading best fine-tuned weights...")

final_model.load_weights(
    str(FINE_TUNE_CHECKPOINT)
)

print("Best fine-tuned weights loaded successfully.")

# ------------------------------------------------------------
# 2. Save complete Keras model
# ------------------------------------------------------------

FINAL_KERAS_MODEL = (
    FINAL_MODEL_DIR /
    "FINAL_DENSENET121_BRISC2025.keras"
)

print("\nSaving complete Keras model...")

final_model.save(
    str(FINAL_KERAS_MODEL)
)

print("Keras model saved successfully.")

# ------------------------------------------------------------
# 3. Verify Keras model exists
# ------------------------------------------------------------

if not FINAL_KERAS_MODEL.exists():
    raise RuntimeError(
        "Final Keras model was not created."
    )

keras_size_mb = (
    FINAL_KERAS_MODEL.stat().st_size /
    (1024 ** 2)
)

print(
    f"Keras model size: "
    f"{keras_size_mb:.2f} MB"
)

# ------------------------------------------------------------
# 4. Save final weights separately
# ------------------------------------------------------------

FINAL_WEIGHTS = (
    FINAL_MODEL_DIR /
    "FINAL_DENSENET121_BRISC2025.weights.h5"
)

print("\nSaving final weights...")

final_model.save_weights(
    str(FINAL_WEIGHTS)
)

if not FINAL_WEIGHTS.exists():
    raise RuntimeError(
        "Final weights file was not created."
    )

weights_size_mb = (
    FINAL_WEIGHTS.stat().st_size /
    (1024 ** 2)
)

print(
    f"Final weights size: "
    f"{weights_size_mb:.2f} MB"
)

# ------------------------------------------------------------
# 5. Export TensorFlow SavedModel
# ------------------------------------------------------------

FINAL_SAVEDMODEL = (
    FINAL_MODEL_DIR /
    "FINAL_DENSENET121_BRISC2025_SavedModel"
)

print("\nExporting TensorFlow SavedModel...")

if FINAL_SAVEDMODEL.exists():
    import shutil
    shutil.rmtree(FINAL_SAVEDMODEL)

tf.saved_model.save(
    final_model,
    str(FINAL_SAVEDMODEL)
)

if not FINAL_SAVEDMODEL.exists():
    raise RuntimeError(
        "TensorFlow SavedModel export failed."
    )

print("TensorFlow SavedModel exported successfully.")

# ------------------------------------------------------------
# 6. Test prediction using current model
# ------------------------------------------------------------

print("\nRunning prediction verification...")

verification_images, verification_labels = next(
    iter(final_train_dataset)
)

verification_predictions = final_model.predict(
    verification_images,
    verbose=0
)

# ------------------------------------------------------------
# 7. Verify prediction output
# ------------------------------------------------------------

print(
    f"Prediction shape: "
    f"{verification_predictions.shape}"
)

if verification_predictions.shape != (
    verification_images.shape[0],
    NUM_CLASSES
):
    raise RuntimeError(
        f"Unexpected prediction shape: "
        f"{verification_predictions.shape}"
    )

# Check probability properties
probability_sums = (
    np.sum(
        verification_predictions,
        axis=1
    )
)

if not np.allclose(
    probability_sums,
    1.0,
    atol=1e-5
):
    raise RuntimeError(
        "Softmax probabilities do not sum to 1."
    )

if (
    np.min(verification_predictions) < 0
    or
    np.max(verification_predictions) > 1
):
    raise RuntimeError(
        "Prediction probabilities outside [0, 1]."
    )

print("Prediction output verification: PASS")

# ------------------------------------------------------------
# 8. Reload the Keras model
# ------------------------------------------------------------

print("\nReloading saved Keras model...")

reloaded_model = keras.models.load_model(
    str(FINAL_KERAS_MODEL),
    compile=False
)

print("Saved Keras model reloaded successfully.")

# ------------------------------------------------------------
# 9. Verify reloaded model prediction
# ------------------------------------------------------------

reloaded_predictions = reloaded_model.predict(
    verification_images,
    verbose=0
)

if reloaded_predictions.shape != verification_predictions.shape:
    raise RuntimeError(
        "Reloaded model prediction shape mismatch."
    )

prediction_difference = np.max(
    np.abs(
        verification_predictions -
        reloaded_predictions
    )
)

print(
    f"Maximum prediction difference after reload: "
    f"{prediction_difference:.12f}"
)

if prediction_difference > 1e-5:
    raise RuntimeError(
        "Reloaded model predictions differ unexpectedly."
    )

print("Reload verification: PASS")

# ------------------------------------------------------------
# 10. Save final model verification record
# ------------------------------------------------------------

final_model_record = pd.DataFrame([{
    "model": "DenseNet121",
    "dataset": "BRISC2025",
    "development_samples": 5000,
    "official_test_samples": 1000,
    "official_test_used": False,
    "initialization": "ImageNet",
    "final_training_stages": "head_training + fine_tuning",
    "keras_model_exists": FINAL_KERAS_MODEL.exists(),
    "weights_exists": FINAL_WEIGHTS.exists(),
    "savedmodel_exists": FINAL_SAVEDMODEL.exists(),
    "prediction_shape_verified": True,
    "probability_output_verified": True,
    "keras_reload_verified": True,
    "prediction_reload_difference": float(
        prediction_difference
    ),
    "status": "PASS"
}])

FINAL_MODEL_RECORD = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_MODEL_VERIFICATION.csv"
)

final_model_record.to_csv(
    FINAL_MODEL_RECORD,
    index=False
)

# ------------------------------------------------------------
# 11. Save final model metadata
# ------------------------------------------------------------

final_metadata = {
    "model": "DenseNet121",
    "dataset": "BRISC2025",
    "development_samples": 5000,
    "official_test_samples": 1000,
    "official_test_used": False,
    "input_size": list(IMAGE_SIZE),
    "channels": NUM_CHANNELS,
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "class_to_index": CLASS_TO_INDEX,
    "initialization": "ImageNet",
    "head_epochs": HEAD_EPOCHS,
    "fine_tune_epochs": FINE_TUNE_EPOCHS,
    "fine_tune_last_n_layers": FINE_TUNE_LAST_N_LAYERS,
    "head_learning_rate": HEAD_LEARNING_RATE,
    "fine_tune_learning_rate": FINE_TUNE_LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "class_weights_used": True,
    "augmentation_used": False,
    "keras_model": str(FINAL_KERAS_MODEL),
    "weights_file": str(FINAL_WEIGHTS),
    "savedmodel_directory": str(FINAL_SAVEDMODEL),
    "prediction_reload_difference": float(
        prediction_difference
    ),
    "status": "PASS",
    "created_at": datetime.now().isoformat()
}

FINAL_METADATA_FILE = (
    FINAL_REPORT_DIR /
    "FINAL_DENSENET121_MODEL_METADATA.json"
)

with open(
    FINAL_METADATA_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_metadata,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 12. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 8 COMPLETE — FINAL MODEL SAVED + VERIFIED")
print("=" * 70)

print("\nFINAL MODEL FILES:")

print(f"\nKeras model:")
print(FINAL_KERAS_MODEL)

print(f"\nWeights:")
print(FINAL_WEIGHTS)

print(f"\nSavedModel:")
print(FINAL_SAVEDMODEL)

print(f"\nVerification record:")
print(FINAL_MODEL_RECORD)

print(f"\nMetadata:")
print(FINAL_METADATA_FILE)

print("\nOfficial test used: NO")
print("Final model verification: PASS")

CELL 8 — SAVE + VERIFY FINAL DENSENET121

Loading best fine-tuned weights...
Best fine-tuned weights loaded successfully.

Saving complete Keras model...
Keras model saved successfully.
Keras model size: 36.84 MB

Saving final weights...
Final weights size: 28.35 MB

Exporting TensorFlow SavedModel...


INFO:tensorflow:Assets written to: D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\models\final_densenet121\FINAL_DENSENET121_BRISC2025_SavedModel\assets


INFO:tensorflow:Assets written to: D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\models\final_densenet121\FINAL_DENSENET121_BRISC2025_SavedModel\assets


TensorFlow SavedModel exported successfully.

Running prediction verification...
Prediction shape: (32, 4)
Prediction output verification: PASS

Reloading saved Keras model...
Saved Keras model reloaded successfully.
Maximum prediction difference after reload: 0.000000000000
Reload verification: PASS

CELL 8 COMPLETE — FINAL MODEL SAVED + VERIFIED

FINAL MODEL FILES:

Keras model:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\models\final_densenet121\FINAL_DENSENET121_BRISC2025.keras

Weights:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\models\final_densenet121\FINAL_DENSENET121_BRISC2025.weights.h5

SavedModel:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\models\final_densenet121\FINAL_DENSENET121_BRISC2025_SavedModel

Verification record:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\outputs\reports\final_densenet121\FINAL_DENSENET121_MODEL_VERIFICATION.csv

Metadata:
D:\MSC Bioinformatics\INTERNSHIP\BrainTumor\outputs\reports\final_densenet121\FINAL_DENSENET121_MODEL_METADATA.json

Offic